# Quantum Optimization for Distributed Order Management (DOM)

# Notebook 05 – Enhanced Classical Optimization

## Objective

This notebook improves the initial optimization model by incorporating additional business constraints and performance metrics.

The enhanced model aims to produce more realistic order assignment decisions while maintaining computational efficiency using Google's OR-Tools.

In [1]:
print("Hello World")

Hello World


## Verify Optimization Environment

Before building the optimization model, the OR-Tools library is verified to ensure that the required optimization solver is available.

In [2]:
import ortools
print(ortools.__version__)

9.15.6755


## Import Required Libraries

The necessary Python libraries for optimization, numerical computation, and data analysis are imported.

In [3]:
from ortools.linear_solver import pywraplp

print("OR-Tools Imported Successfully!")

OR-Tools Imported Successfully!


In [4]:
import pandas as pd
import numpy as np
from ortools.linear_solver import pywraplp

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


# Load Prepared Datasets

The cleaned order dataset and supporting operational datasets are loaded.

These datasets describe customer demand, warehouse capacity, shipping costs, dock limitations, and throughput constraints required for optimization.

In [6]:
orders = pd.read_csv("../data/orders_clean.csv")
capacity = pd.read_csv("../data/input data/input_capacity_planning.csv")
shipping = pd.read_csv("../data/input data/input_shipping_cost_data.csv")
dock = pd.read_csv("../data/input data/input_dock_capacity.csv")
throughput = pd.read_csv("../data/input data/input_throughput_capacity.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


## Verify Dataset Dimensions

The dimensions of all datasets are verified before optimization begins.

This confirms that the required input data has been loaded successfully.

In [7]:
print("Orders:", orders.shape)
print("Capacity:", capacity.shape)
print("Shipping:", shipping.shape)
print("Dock:", dock.shape)
print("Throughput:", throughput.shape)

Orders: (25193, 36)
Capacity: (377504, 23)
Shipping: (12922, 7)
Dock: (480, 13)
Throughput: (530, 7)


## Select Optimization Sample

A subset of customer orders is selected to create a tractable optimization problem.

Working with a representative sample allows faster experimentation while preserving the structure of the original dataset.

In [8]:
sample_orders = orders.head(100).copy()

print("Sample Orders:", sample_orders.shape)

Sample Orders: (100, 36)


# Create Optimization Model

The SCIP solver provided by Google's OR-Tools is used to formulate the optimization problem.

The solver searches for the best feasible solution that satisfies all defined constraints.

In [9]:
solver = pywraplp.Solver.CreateSolver("SCIP")

print("Solver Created Successfully!")

Solver Created Successfully!


## Decision Variables

Binary decision variables are created for every customer order.

A value of **1** indicates that an order is selected for fulfillment, while **0** indicates that it is not selected.

In [10]:
x = {}

for i in sample_orders.index:
    x[i] = solver.BoolVar(f"x_{i}")

print("Decision Variables Created:", len(x))

Decision Variables Created: 100


## Business Constraints

Business constraints are added to ensure that only feasible order assignments are considered.

The optimization respects inventory availability and operational limitations.

In [11]:
for i in sample_orders.index:
    if sample_orders.loc[i, "IsInvAvail"] == "N":
        solver.Add(x[i] == 0)

print("Inventory Constraints Added Successfully!")

Inventory Constraints Added Successfully!


## Capacity Constraints

Additional constraints limit the total number of selected orders.

These constraints simulate the limited operational capacity of Distribution Centers.

In [12]:
solver.Add(
    solver.Sum(x[i] for i in sample_orders.index) <= 80
)

print("Capacity Constraint Added Successfully!")

Capacity Constraint Added Successfully!


## Objective Function

The objective function maximizes the business value obtained from fulfilling customer orders.

The optimization attempts to select the most valuable set of feasible assignments.

In [13]:
solver.Maximize(
    solver.Sum(
        (sample_orders.loc[i, "Order_SKU_Revenue"]
         - 0.1 * sample_orders.loc[i, "OrderedWeight"]) * x[i]
        for i in sample_orders.index
    )
)

print("Objective Function Added Successfully!")

Objective Function Added Successfully!


## Solve the Optimization Model

The optimization model is executed using the SCIP solver.

The solver evaluates all constraints and returns the best feasible solution.

In [14]:
status = solver.Solve()

print("Solver Status:", status)

Solver Status: 0


### Observation

The optimization solver successfully identifies the best feasible order assignment for the selected dataset.

## Store Optimization Results

The selected optimization decisions are added to the dataset.

These assignments will be used for evaluation and comparison with baseline methods.

In [15]:
sample_orders["Selected"] = [
    int(x[i].solution_value())
    for i in sample_orders.index
]

sample_orders.head()

,Group_Flag,Plant,MaterialNumber,transportationplanningdate,IsTopCust,OpeningStock,RequestedDeliveryDate,DeliveryNoteFlag,IsInvAvail,LoadNumber,...,Measure,FillRateThreshold,Penaltyforpotentialcuts,MaximumPenalty,FixedPenalty,FixedPenaltyPerSKU,MinimumPenalty,OnTimePercentage,OnTimeFixed,Selected
0,5484913123,5083,12260382,6/26/24,N,33814.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,5484913123,5083,9516458,6/26/24,N,6546.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,5484913123,5083,12408924,6/26/24,N,4151.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,5484913123,5083,9517630,6/26/24,N,17667.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,5484913123,5083,12587091,6/26/24,N,89341.0,6/27/24,N,Y,U600105191,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


## Performance Evaluation

Several performance indicators are calculated to evaluate the optimization results.

These metrics include the number of fulfilled orders, fill rate, revenue, and shipment weight.

In [16]:
selected = sample_orders["Selected"].sum()

print("Selected Orders:", selected)
print("Rejected Orders:", len(sample_orders) - selected)

Selected Orders: 80
Rejected Orders: 20


## Fill Rate

The fill rate measures the percentage of customer orders successfully selected for fulfillment.

Higher fill rates generally indicate better customer service performance.

In [17]:
fill_rate = (selected / len(sample_orders)) * 100

print(f"Fill Rate: {fill_rate:.2f}%")

Fill Rate: 80.00%


## Revenue Analysis

The total revenue generated by the selected customer orders is calculated.

Revenue is an important business metric for evaluating optimization quality.

In [18]:
total_revenue = sample_orders.loc[
    sample_orders["Selected"] == 1,
    "Order_SKU_Revenue"
].sum()

print("Total Revenue:", total_revenue)

Total Revenue: 283497


## Shipment Weight Analysis

The total shipment weight of selected orders is calculated.

Shipment weight helps estimate transportation workload and logistics requirements.

In [19]:
total_weight = sample_orders.loc[
    sample_orders["Selected"] == 1,
    "OrderedWeight"
].sum()

print("Total Shipping Weight:", total_weight)

Total Shipping Weight: 97088.70300000001


## Save Optimized Results

The optimized order assignments are saved for use in subsequent notebooks.

These results will later be compared with advanced optimization and quantum approaches.

In [20]:
sample_orders.to_csv(
    "../data/optimized_orders_v2.csv",
    index=False
)

print("Optimization Version 2 Results Saved Successfully!")

Optimization Version 2 Results Saved Successfully!


# Conclusion

This notebook enhanced the classical optimization model by introducing additional business constraints and evaluating multiple performance metrics.

The optimized solution demonstrates improved decision-making compared to the simple baseline while remaining computationally efficient.

The results generated here provide a stronger benchmark for the advanced optimization and quantum models developed in the following notebooks.